In [1]:
import sys
import os

import Levenshtein

from datasets import load_dataset
import pandas as pd
import evaluate
import json
import re
import numpy as np

from sentence_transformers import SentenceTransformer, util

# language_wise_training = '<language>'
dataset1 = load_dataset('Vikir2411CS19/IPRET-V1',split='train')

dataset1 = dataset1.add_column("language_encoded", dataset1["language"])
dataset1 = dataset1.class_encode_column("language_encoded")
# dataset1 = dataset1.filter(lambda row: row['language']==language_wise_training, num_proc=4)
dataset1 = dataset1.filter(lambda row: row['language']=='tamil', num_proc=4)

# dataset_split = dataset.train_test_split(test_size=0.3, seed=42, stratify_by_column="language_encoded")
split_dataset1 = dataset1.train_test_split(test_size=0.3, seed=42)


train_dataset1 = split_dataset1["train"]
test_dataset1 = split_dataset1["test"]

format_rewards = []
accuracy_rewards=[]
inconsis_penaltys=[]
tool_debate_rewards=[]
partial_rewards = []
tool_count = []



In [2]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a MCQ question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the correct option. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> correct option here </answer>"
)

In [3]:
def format_helper_userinput(example):
    # def format_helper(example):
    # Language-specific option label mapping
    option_label_map = {
        "english": {
            "a":"a",
            "b":"b",
            "c":"c",
            "d":"d"
        },
        "hindi": {
            "a": "क",
            "b": "ख",
            "c": "ग",
            "d": "घ"
        },
        "bengali": {
            "a": "ক",
            "b": "খ",
            "c": "গ",
            "d": "ঘ"
        },
        "marathi": {
            "a": "क",
            "b": "ख",
            "c": "ग",
            "d": "घ"
        },
        "gujarati": {
            "a": "ક",
            "b": "ખ",
            "c": "ગ",
            "d": "ઘ"
        },
        "tamil": {
            "a": "௧",
            "b": "௨",
            "c": "௩",
            "d": "௪"
        }
    }

    # Normalize language and answer key
    language = example.get("language", "").strip().lower()
    # answer_key = example.get("Final Answer", "").strip().lower()  # a/b/c/d
    native_answer = example.get("final_answer")
    lang_map = option_label_map.get(language, {})
    reverse_map = {v: k for k, v in lang_map.items()}  # native → a/b/c/d

    # Resolve option key
    try:
        option_key_letter = reverse_map.get(native_answer)
        option_text = (
            example.get(f"option_{option_key_letter}")
            if option_key_letter else None
        )
    except:
        option_key_letter = native_answer
        option_text = native_answer

    

    user_input = (
        f"Question: {example['question']}\n"
        f"Options: "
        f"{lang_map.get('a','a')}. {example['option_a']}, "
        f"{lang_map.get('b','b')}. {example['option_b']}, "
        f"{lang_map.get('c','c')}. {example['option_c']}, "
        f"{lang_map.get('d','d')}. {example['option_d']}."
    )

    return {
        "user_input": user_input,
        "native_answer_label": native_answer,
        "option_key": option_key_letter,
        "answer_text": option_text
    }

def format_data1(example):
    # templist = []
    result_format_helper = format_helper_userinput(example)
    user_input = result_format_helper['user_input']
    # if example['final_answer']==None:
    #     sol = f"<think>{example['Reasoning']}</think><answer>None</answer>"
    try:
        sol = (
        f"<think> None </think>"
        f"<answer>{result_format_helper['native_answer_label']}. {result_format_helper['answer_text']}</answer>"
        )
    except:
        sol = f"<think> None </think> <answer> {example['final_answer']} </answer>"

    returned_dict = [
          {
              "role": "system",
              "content": SYSTEM_PROMPT,
          },
          {
              "role": "user",
              "content": user_input,
          },
      ]

    return {
        # "images":[],
      "prompt": returned_dict,
        "solution":sol,
      }

In [ ]:
format_data1(dataset1[1])

In [5]:
train_dataset=[format_data1(example) for example in train_dataset1]


In [ ]:
import torch
from transformers import AutoModelForCausalLM,AutoProcessor,AutoTokenizer 
model_id = "Qwen/Qwen2.5-3B-Instruct"
processor = AutoProcessor.from_pretrained(model_id, use_fast=True, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    # use_cache= False
)

sentence_transformer_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


In [9]:
import re
def format_reward(completions, solution, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    # pattern = r"^<think>.*?</think>\s*<answer>.*?</answer><think>.*?</think>\s*<answer>.*?</answer>$"
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"

    matches = [re.match(pattern, content[0]['content'], re.DOTALL | re.MULTILINE) for content in completions]
    rewards = [1.0 if match else 0.0 for match in matches]
    format_rewards.append(max(rewards))
    return rewards


In [10]:
def accuracy_reward(completions, solution, **kwargs):
    rewards = []
    pattern = r"<think>(.*?)</think>\s*<answer>(.*?)</answer>"
    for completion, sol in zip(completions, solution):
        # print("SOLUTION:",sol)
        # print("COMPLETION:", completion)
        sol_match = re.search(pattern, sol, flags=re.DOTALL | re.MULTILINE)
        gt_answer='None'
        pred_answer='None'
        if sol_match:
            gt_answer = sol_match.group(2).strip().lower()
        match=re.search(pattern, completion[0]['content'], flags=re.DOTALL | re.MULTILINE)
        if match:
            pred_answer=match.group(2).strip().lower()
        if gt_answer==pred_answer:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    # print(rewards)
    accuracy_rewards.append(max(rewards))

    return rewards

In [11]:
def sentence_similarity(sent1,sent2):
    cosine_score = util.cos_sim(sentence_transformer_model.encode(sent1, convert_to_tensor=True), sentence_transformer_model.encode(sent2, convert_to_tensor=True)).item()
    levi_score = Levenshtein.ratio(sent1,sent2)
    # print(levi_score)
    sim_score = (cosine_score+levi_score)/2
    return sim_score

In [12]:
def partial_reward(completions, solution, **kwargs):
    rewards = []
    pattern = r"<think>(.*?)</think>\s*<answer>(.*?)</answer>"
    for completion, sol in zip(completions, solution):
        # print("SOLUTION:",sol)
        # print("COMPLETION:", completion)
        sol_match = re.search(pattern, sol, flags=re.DOTALL | re.MULTILINE)
        gt_answer='None'
        pred_answer='None'
        if sol_match:
            gt_answer = sol_match.group(2).strip().lower()
        match=re.search(pattern, completion[0]['content'], flags=re.DOTALL | re.MULTILINE)
        if match:
            pred_answer=match.group(2).strip().lower()

        sent_sim = sentence_similarity(gt_answer, pred_answer)

        if sent_sim == 1:
            rewards.append(1.0)
        else:
            rewards.append(sent_sim-1)
    # print(rewards)
    partial_rewards.append(max(rewards))
    # print(rewards)
    return rewards
            
        

In [13]:
from trl import GRPOConfig

# Configure training arguments using GRPOConfig
training_args = GRPOConfig(
    output_dir="BPR_text-Qwen2.5-3B-mcq_tam",
    learning_rate=1e-5,
    remove_unused_columns=False,  # to access the solution column in accuracy_reward
    num_train_epochs=3,
    bf16=True,
    # Parameters that control the data preprocessing
    # per_device_train_batch_size=16,
    per_device_train_batch_size=16,

    max_completion_length=256,  # default: 256
    num_generations=2,  # default: 8
    # num_generations=8,  # default: 8
    # generation_batch_size=8,
    # max_prompt_length=2048,
    max_prompt_length=10000,
    report_to='none',
    logging_steps=10,
    push_to_hub=False,
    save_strategy="epoch",
)

In [ ]:
from trl import GRPOTrainer
import json

trainer = GRPOTrainer(
    model=model,
    processing_class=processor,
    reward_funcs=[format_reward,accuracy_reward, partial_reward],
    args=training_args,
    train_dataset=train_dataset,
)

In [ ]:
trainer.train()


In [ ]:
trainer.save_model(training_args.output_dir)


In [ ]:
print('done')